# A Small Drawing Language — Warped Grid Case Study

This notebook shows one piece built from the `math_vis` drawing-language engine: a field-warped grid whose colour and geometry are driven by the same smooth field.

The language treats a drawing as data:

$$\text{layout} \rightarrow \text{columns} \rightarrow \text{shape} \rightarrow \text{style} \rightarrow \text{color}$$

All random draws are seeded by `(master, layer, column, k)`, so editing one column does not change any other.

In [ ]:
%matplotlib inline
import zlib
from dataclasses import dataclass, field as dc_field, replace

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection, LineCollection
from matplotlib.colors import to_rgb
from PIL import Image
from scipy.ndimage import gaussian_filter1d

from math_vis import smooth_field, sample, make_cmap, cosine_cmap, finish, fig_to_img

# ---------- parameters ----------
MASTER = 7          # master seed: vary this to curate, then freeze it
PAPER  = '#f1e8d6'  # warm paper, never pure white
A      = 1.25       # canvas is 1 unit wide and A units tall; y grows downward
QUALITY = dict(dpi=90, ss=2)   # lower for quick tries; raise dpi for a print

In [ ]:
def stream(master, *key):
    # Independent RNG for (master, layer, column, k).
    words = [master] + [zlib.crc32(str(k).encode()) for k in key]
    return np.random.default_rng(np.random.SeedSequence(words))

a = stream(MASTER, 'demo', 'a', 1).uniform(size=3)
b = stream(MASTER, 'demo', 'a', 1).uniform(size=3)
c = stream(MASTER, 'demo', 'b', 1).uniform(size=3)
print('same key     :', a.round(3), b.round(3))
print('different key:', c.round(3))
assert np.array_equal(a, b) and not np.array_equal(a, c)

In [ ]:
class Ctx:
    # What an expression can see: the table and a seeded stream per random operator.
    def __init__(self, master, layer, table, fields):
        self.master, self.layer, self.t, self.fields = master, layer, table, fields
        self.n = len(table['x']); self.column = None; self._k = 0
    def begin(self, column):            # start evaluating a new column
        self.column, self._k = column, 0
    def rng(self):                      # next random stream for this column
        self._k += 1
        return stream(self.master, self.layer, self.column, self._k)
    def group(self, per):
        return np.zeros(self.n, int) if per == 'all' else self.t[per]

class Expr:
    def __init__(self, desc, fn): self.desc, self.fn = desc, fn
    def __call__(self, ctx): return self.fn(ctx)
    def __repr__(self): return self.desc
    def _bin(self, o, f, sym, rev=False):
        o = ex(o); l, r = (o, self) if rev else (self, o)
        return Expr(f'({l.desc} {sym} {r.desc})', lambda c: f(l(c), r(c)))
    __add__  = lambda s, o: s._bin(o, np.add, '+')
    __radd__ = lambda s, o: s._bin(o, np.add, '+', True)
    __sub__  = lambda s, o: s._bin(o, np.subtract, '-')
    __rsub__ = lambda s, o: s._bin(o, np.subtract, '-', True)
    __mul__  = lambda s, o: s._bin(o, np.multiply, '*')
    __rmul__ = lambda s, o: s._bin(o, np.multiply, '*', True)
    __truediv__ = lambda s, o: s._bin(o, np.divide, '/')

def ex(v):
    return v if isinstance(v, Expr) else Expr(repr(v), lambda c: np.full(c.n, v))

def U(a, b):      return Expr(f'U({a},{b})',      lambda c: c.rng().uniform(a, b, c.n))
def N(mu, sd):    return Expr(f'N({mu},{sd})',    lambda c: c.rng().normal(mu, sd, c.n))
def col(name):    return Expr(f'col({name})',     lambda c: c.t[name])
def coord(axis):  return Expr(f'coord({axis})',   lambda c: c.t[axis])

def field(name):
    def fn(c):
        f = c.fields[name]; gw = f.shape[1]          # field grid cell = 1/gw canvas units
        return sample(f, c.t['y'] * gw, c.t['x'] * gw)
    return Expr(f'field({name})', fn)

def _by_group(vals, g, f):
    out = np.empty(len(vals), float)
    for gid in np.unique(g):
        m = g == gid
        out[m] = f(vals[m])
    return out

def walk(step, per='row'):
    step = ex(step)
    return Expr(f'walk({step}, per={per})', lambda c: _by_group(step(c), c.group(per), np.cumsum))

def smooth(e, width, per='row'):
    e = ex(e)
    return Expr(f'smooth({e}, {width}, per={per})',
                lambda c: _by_group(e(c), c.group(per), lambda v: gaussian_filter1d(v, width, mode='nearest')))

def choice(values, weights):
    p = np.asarray(weights, float); p = p / p.sum()
    return Expr(f'choice({list(values)}, {list(weights)})',
                lambda c: c.rng().choice(np.asarray(values), size=c.n, p=p))

def ramp(e, lo, hi, out=(0.0, 1.0)):
    e = ex(e)
    return Expr(f'ramp({e}, {lo}..{hi} -> {out[0]}..{out[1]})',
                lambda c: out[0] + (out[1] - out[0]) * np.clip((e(c) - lo) / (hi - lo), 0, 1))

In [ ]:
@dataclass
class Rows:
    ys: object; x0: float; x1: float; advance: Expr
    pre: dict = dc_field(default_factory=dict); n_max: int = 400

@dataclass
class Warp:
    field: str; amp: float; angle: object = 0.0    # radians, or the name of a field (angle = pi * field)

@dataclass
class Grid:
    nx: int; ny: int; x0: float = 0.0; x1: float = 1.0; y0: float = 0.0; y1: float = A
    warps: tuple = ()

def make_fields(decls, master, aspect=A):
    # decls: {name: (sigma, grid_width)}; each field gets its own isolated stream.
    out = {}
    for name, (sigma, gw) in decls.items():
        out[name] = smooth_field((int(round(gw * aspect)), gw), sigma, stream(master, 'field', name))
    return out

def _finish_table(t, u, v):
    t['i'] = np.arange(len(t['x'])); t['u'] = u; t['v'] = v
    return t

def _rows(L, name, master, fields):
    ys = np.asarray(L.ys, float); R = len(ys); n = R * L.n_max
    t = {'row': np.repeat(np.arange(R), L.n_max), 'y': np.repeat(ys, L.n_max), 'x': np.zeros(n)}
    ctx = Ctx(master, name, t, fields)
    for cname, e in L.pre.items():
        ctx.begin(cname); t[cname] = np.asarray(ex(e)(ctx))
    ctx.begin('_advance'); adv = np.asarray(L.advance(ctx), float)
    x = np.empty(n)
    for r in range(R):                                   # x = start + sum of earlier advances
        s = slice(r * L.n_max, (r + 1) * L.n_max)
        x[s] = L.x0 + np.concatenate([[0.0], np.cumsum(adv[s])[:-1]])
    if (x.reshape(R, -1)[:, -1] <= L.x1).any():
        raise ValueError(f"layer '{name}': n_max={L.n_max} too small to reach x1; raise it")
    keep = x <= L.x1
    t = {k: v[keep] for k, v in t.items()}; t['x'] = x[keep]
    t['col'] = np.concatenate([np.arange(c) for c in np.bincount(t['row'], minlength=R)])
    span = ys.max() - ys.min()
    return _finish_table(t, (t['x'] - L.x0) / (L.x1 - L.x0), (t['y'] - ys.min()) / (span if span else 1))

def grid_vertices(L, fields):
    X, Y = np.meshgrid(np.linspace(L.x0, L.x1, L.nx + 1), np.linspace(L.y0, L.y1, L.ny + 1))
    for w in L.warps:                                    # chained: each warp sees the previous result
        f = fields[w.field]; gw = f.shape[1]
        val = sample(f, Y * gw, X * gw)
        if isinstance(w.angle, str):
            g = fields[w.angle]; ang = np.pi * sample(g, Y * gw, X * gw)
        else:
            ang = w.angle
        X, Y = X + w.amp * val * np.cos(ang), Y + w.amp * val * np.sin(ang)
    return X, Y

def _grid(L, name, master, fields):
    X, Y = grid_vertices(L, fields)
    quad = np.stack([np.stack([X[:-1, :-1], Y[:-1, :-1]], -1), np.stack([X[:-1, 1:], Y[:-1, 1:]], -1),
                     np.stack([X[1:, 1:], Y[1:, 1:]], -1),     np.stack([X[1:, :-1], Y[1:, :-1]], -1)], axis=2)
    quad = quad.reshape(-1, 4, 2); c = quad.mean(1)
    t = {'x': c[:, 0], 'y': c[:, 1], 'quad': quad,
         'row': np.repeat(np.arange(L.ny), L.nx), 'col': np.tile(np.arange(L.nx), L.ny)}
    return _finish_table(t, t['col'] / max(L.nx - 1, 1), t['row'] / max(L.ny - 1, 1))

def make_table(layout, name, master, fields):
    if isinstance(layout, Rows): return _rows(layout, name, master, fields)
    if isinstance(layout, Grid): return _grid(layout, name, master, fields)
    raise TypeError(f"layer '{name}': unknown layout {type(layout).__name__}")

In [ ]:
def fold_report(layer, fields, master=None, silent=False):
    """Return (folded%, pinched%) for a Grid layout.

    `fields` may be either the declaration dict passed to `render` or the
    already-built field arrays.  Folded = signed area <= 0.  Pinched = signed
    area < 25% of the undistorted cell area.
    """
    L = layer.layout
    if not isinstance(L, Grid):
        if not silent:
            print(f"{layer.name}: not a grid")
        return 0.0, 0.0
    # If declarations were passed, build the actual fields.
    if isinstance(fields, dict) and isinstance(next(iter(fields.values())), tuple):
        fields = make_fields(fields, master, A)
    X, Y = grid_vertices(L, fields)
    q = np.stack([np.stack([X[:-1, :-1], Y[:-1, :-1]], -1),
                  np.stack([X[:-1, 1:],  Y[:-1, 1:]],  -1),
                  np.stack([X[1:, 1:],   Y[1:, 1:]],   -1),
                  np.stack([X[1:, :-1],  Y[1:, :-1]],  -1)], axis=2)
    x = q[..., 0]
    y = q[..., 1]
    signed = 0.5 * np.sum(x * np.roll(y, -1, axis=-1) - np.roll(x, -1, axis=-1) * y, axis=-1)
    nominal = (L.x1 - L.x0) * (L.y1 - L.y0) / (L.nx * L.ny)
    n = signed.size
    folded = (signed <= 0).sum() / n * 100
    pinched = (signed < 0.25 * nominal).sum() / n * 100
    if not silent:
        print(f"{layer.name}: {folded:.1f}% folded, {pinched:.1f}% pinched")
    return folded, pinched


In [ ]:
@dataclass
class Layer:
    name: str
    layout: object
    columns: dict = dc_field(default_factory=dict)   # name -> Expr, evaluated in order
    shape:   dict = dc_field(default_factory=dict)
    style:   dict = dc_field(default_factory=dict)
    color:   dict = dc_field(default_factory=dict)
    order:   tuple = None                             # e.g. ('y', 'asc'): painter order inside the layer

def evaluate(layer, master, fields):
    t = make_table(layer.layout, layer.name, master, fields)
    ctx = Ctx(master, layer.name, t, fields)
    for cname, e in layer.columns.items():
        ctx.begin(cname)
        try:
            t[cname] = np.asarray(ex(e)(ctx))
        except KeyError as err:
            raise KeyError(f"layer '{layer.name}', column '{cname}': unknown column {err}") from None
    if layer.order:                                   # note: walks above already ran in layout order
        key, direction = layer.order
        idx = np.argsort(t[key], kind='stable')
        idx = idx if direction == 'asc' else idx[::-1]
        ctx.t = {k: v[idx] for k, v in t.items()}
    return ctx

def param(ctx, group, spec, key, default):
    ctx.begin(f'{group}.{key}')
    return np.asarray(ex(spec.get(key, default))(ctx))

def geometry(layer, ctx):
    # Returns anchor (E,2) and polygon points relative to it (E,P,2).
    t, sh = ctx.t, layer.shape
    dx, dy = param(ctx, 'shape', sh, 'dx', 0.0), param(ctx, 'shape', sh, 'dy', 0.0)
    kind = sh['kind']
    if kind == 'quad':
        c = t['quad'].mean(1)
        return c + np.column_stack([dx, dy]), t['quad'] - c[:, None, :]
    r = param(ctx, 'shape', sh, 'r', 0.02)
    anchor = np.column_stack([t['x'] + dx, t['y'] + dy])
    if kind == 'circle':
        ang = np.linspace(0, 2 * np.pi, 40, endpoint=False)[None, :]
    elif kind == 'halfdisc':
        orient = param(ctx, 'shape', sh, 'orient', -np.pi / 2)   # dome direction; -pi/2 = up on screen
        ang = np.linspace(-np.pi / 2, np.pi / 2, 32)[None, :] + orient[:, None]
    else:
        raise ValueError(f"layer '{layer.name}': unknown shape '{kind}'")
    return anchor, r[:, None, None] * np.stack([np.cos(ang), np.sin(ang)], -1)

def layer_items(layer, ctx, bg):
    anchor, rel = geometry(layer, ctx)
    E, P, _ = rel.shape
    st, co = layer.style, layer.color

    tt = np.clip(param(ctx, 'color', co, 't', 0.5), 0, 1)
    rgb = np.array(co['palette'](tt)[:, :3])
    for k, (mask, hexc) in enumerate(co.get('accents', [])):
        ctx.begin(f'color.accent{k}')
        rgb[ex(mask)(ctx) > 0.5] = to_rgb(hexc)

    state = param(ctx, 'style', st, 'fill', 'fill')
    alpha = np.clip(param(ctx, 'style', st, 'alpha', 1.0), 0, 1)
    lw    = param(ctx, 'style', st, 'stroke', 0.4)
    filled, masked = state == 'fill', state == 'mask'
    outlined = filled | (state == 'outline')

    sd = st.get('shade'); N = sd['layers'] if sd else 0
    scales = np.concatenate([[1.0], 1 - sd['spread'] * np.arange(1, N + 1) / N if N else [], [1.0]])
    M = N + 2                                         # per element: fill, N shade copies, outline

    verts = anchor[:, None, None, :] + rel[:, None] * scales[None, :, None, None]
    face = np.zeros((E, M, 4)); edge = np.zeros((E, M, 4)); lws = np.zeros((E, M))
    face[:, 0, :3] = np.where(masked[:, None], to_rgb(bg), rgb)
    face[:, 0, 3]  = np.where(filled, alpha, np.where(masked, 1.0, 0.0))
    if N:
        face[:, 1:N + 1, :3] = (rgb * sd.get('tint', 0.35))[:, None, :]   # dark tint of the fill, never black
        face[:, 1:N + 1, 3]  = np.where(filled, sd['alpha'], 0.0)[:, None]
    edge[:, -1, :3] = rgb * st.get('stroke_dark', 0.55)                   # outline: darker than fill
    edge[:, -1, 3]  = np.where(outlined, 0.9, 0.0)
    lws[:, -1]      = np.where(outlined, lw, 0.0)
    return PolyCollection(verts.reshape(-1, P, 2), facecolors=face.reshape(-1, 4),
                          edgecolors=edge.reshape(-1, 4), linewidths=lws.ravel(), antialiaseds=True)

def render(layers, fields=None, master=MASTER, bg=PAPER, aspect=A, width_in=6.0,
           dpi=90, ss=2, grain=3.0, vignette=0.0):
    # Draw layers in order at dpi*ss, downsample by ss (antialiasing), add grain/vignette.
    fdict = make_fields(fields or {}, master, aspect)
    fig = plt.figure(figsize=(width_in, width_in * aspect), dpi=dpi * ss)
    fig.patch.set_facecolor(bg)
    ax = fig.add_axes([0, 0, 1, 1]); ax.set_xlim(0, 1); ax.set_ylim(aspect, 0); ax.axis('off')
    tables = {}
    for L in layers:
        ctx = evaluate(L, master, fdict)
        ax.add_collection(layer_items(L, ctx, bg))
        tables[L.name] = ctx.t
    img = fig_to_img(fig, dpi=dpi * ss)
    if ss > 1:
        img = np.asarray(Image.fromarray(img).resize((img.shape[1] // ss, img.shape[0] // ss), Image.LANCZOS))
    return finish(img, grain=grain, vignette=vignette, seed=master, display=False), tables

def show(img, width=480):
    finish(img, grain=0, display=True, width=width)

def dump(t, n=5):
    # Inspect an element table (non-array columns skipped).
    keys = [k for k, v in t.items() if v.ndim == 1]
    print(' | '.join(f'{k:>8}' for k in keys))
    for r in range(n):
        print(' | '.join(f'{t[k][r]:>8.3f}' if t[k].dtype.kind == 'f' else f'{t[k][r]!s:>8}' for k in keys))
    print(f'... {len(t["x"])} elements')

## Piece: warped grid (mercury)

The grid corners are displaced by the field `form` (and a finer chained warp). Colour comes from that *same* field, so shape and colour move together. A second field, `wire`, drives stroke width and transparency.

In [ ]:
# Palette options. Change the palette name in SETTINGS to try another.
MERC_V1 = make_cmap(['#22313f', '#4a6a7a', '#9fb3b0', '#e6dcc3', '#d8a25e', '#b5523b'])
SLATE = make_cmap(['#2a3b45', '#4a6270', '#7a929e', '#a8bcc4', '#d5e0e3'])
OLIVE = make_cmap(['#3a4030', '#5e6b4e', '#8d9b78', '#c2b896', '#eaddc5'])
INDIGO_GOLD = make_cmap(['#2e2a4d', '#4a4a6a', '#7a7a8a', '#b8a878', '#d8c060'])
DEEP_RUST = make_cmap(['#3d1e1b', '#6d3a32', '#9e6050', '#c99b82', '#f0d4b8'])
TIDE = cosine_cmap(a=(0.266, 0.239, 0.388),
                   b=(-0.194, 0.242, -0.004),
                   c=(0.395, 0.297, 0.889),
                   d=(0.119, 0.656, 0.976))

PALETTES = {
    'MERC_V1': MERC_V1,
    'SLATE': SLATE,
    'OLIVE': OLIVE,
    'INDIGO_GOLD': INDIGO_GOLD,
    'DEEP_RUST': DEEP_RUST,
    'TIDE': TIDE,
}

# Chosen palette
MERC = INDIGO_GOLD

# Field declarations: larger sigmas than the original so the warp stays smooth.
FIELDS = {'form': (28, 200), 'dir': (42, 200), 'wire': (9, 200), 'fine': (12, 200)}


def make_mercury(fields=FIELDS, nx=88, ny=110, amp_form=0.014, amp_fine=0.0028,
                 palette=MERC, jitter=0.01, none_w=0.0, outline_w=0.0,
                 stroke_dark=0.8, sw_base=(0.15 * 44 / 88, 1.1 * 44 / 88),
                 t_range=(-2.2, 2.2)):
    """Build a mercury layer. Defaults are the chosen settings."""
    fill_w = [1.0 - outline_w - none_w, outline_w, none_w]
    return Layer(
        name='mercury',
        layout=Grid(nx=nx, ny=ny, x0=-0.06, x1=1.06, y0=-0.06, y1=A + 0.06,
                    warps=(Warp('form', amp_form, 'dir'),
                           Warp('fine', amp_fine, 'dir'))),
        columns={'t': ramp(field('form'), t_range[0], t_range[1]) + N(0, jitter),
                 'sw': ramp(field('wire'), -2, 2, out=sw_base),
                 'al': ramp(field('wire'), -2.5, 2.5, out=(0.70, 1.0))},
        shape=dict(kind='quad'),
        style=dict(fill=choice(['fill', 'outline', 'none'], fill_w),
                   alpha=col('al'), stroke=col('sw'),
                   stroke_dark=stroke_dark),
        color=dict(t=col('t'), palette=palette),
    )


# Experimentation block: change any value here.
scale_sw = lambda nx: (0.15 * 44 / nx, 1.1 * 44 / nx)

SETTINGS = {
    'fields': FIELDS,
    'nx': 88, 'ny': 110,
    'amp_form': 0.014, 'amp_fine': 0.0028,
    'palette': MERC,                # try PALETTES['SLATE'], PALETTES['OLIVE'], etc.
    'jitter': 0.01,
    'none_w': 0.0,
    'outline_w': 0.0,
    'stroke_dark': 0.8,
    'sw_base': scale_sw(88),
}

mercury = make_mercury(**SETTINGS)

img_merc, tabs = render([mercury], fields=SETTINGS['fields'], **QUALITY)
show(img_merc)
fold_report(mercury, SETTINGS['fields'], MASTER)

### Palette explorer

The same warp rendered with each palette. Switch the active palette by changing `SETTINGS['palette']` in the cell above.

In [ ]:
def quick_preview(palette, dpi=50, ss=1):
    layer = make_mercury(fields=FIELDS, palette=palette,
                         nx=88, ny=110, amp_form=0.014, amp_fine=0.0028,
                         jitter=0.01, none_w=0.0, outline_w=0.0,
                         stroke_dark=0.8, sw_base=scale_sw(88))
    img, _ = render([layer], fields=FIELDS, dpi=dpi, ss=ss)
    return img

fig, axs = plt.subplots(2, 3, figsize=(12, 8))
for ax, (name, pal) in zip(axs.ravel(), PALETTES.items()):
    ax.imshow(quick_preview(pal))
    ax.set_title(name)
    ax.axis('off')
plt.suptitle('Palette previews (low-res, same seed, same warp)', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

### Colour and warp are dependent

Look at the layer definition:

```python
columns={'t': ramp(field('form'), ...) + N(0, jitter)}
```

The same `field('form')` that pushes the grid corners (`Warp('form', ...)`) also selects the colour. If you change the `form` field, both the warping and the colour bands move together. The jitter adds noise on top, but the large-scale colour pattern follows the warp ridges.

### How this piece relates to the archived notebook

`archive/demo_drawing_language_exploration.ipynb` also builds two row-based pieces (shaded arcs and raging surf). All three pieces share the same engine but are otherwise independent compositions. The relation is vocabulary, not theme: they show the same `Layer` pipeline applied to different layouts (`Rows` vs `Grid`) and shapes.

### Decisions baked in

- Smoother fields and reduced amplitude eliminated folded cells (target: 0% folded, <0.5% pinched).
- Finer grid (`88 × 110`) with scaled strokes keeps line weight consistent.
- No holes or outline cells: every cell is filled.
- `stroke_dark=0.8` keeps edges from turning muddy.